In [1]:
import pandas as pd
import numpy as np

In [2]:
# Importing the dataset

df = pd.read_csv("synthetic_it_support_tickets.csv")
df.head()

,ticket_id,created_at,customer_id,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,has_attachment,platform,region
0,TCKT_000001,2024-01-31T05:14:27,CUST_00861,individual,email,data_export,account_access,low,resolved,standard,I cannot log in; the system says my password i...,Sorry to hear you're having trouble accessing ...,Reset account credentials and confirmed succes...,36.53,0,very_negative,1,0,android,EU
1,TCKT_000002,2024-10-20T06:15:49,CUST_00770,individual,in_app,billing,security_concern,medium,closed_no_action,standard,I noticed a suspicious login on my account.,We take security very seriously. Our team is r...,Ticket closed without further action after no ...,238.32,0,neutral,3,0,web,NaN
2,TCKT_000003,2024-06-18T21:35:54,CUST_02559,small_business,chat,api_integration,bug,low,in_progress,standard,The api integration feature is not saving my c...,Thanks for reporting this bug. We will look in...,NaN,NaN,0,neutral,3,0,android,MEA
3,TCKT_000004,2025-12-25T15:59:52,CUST_03557,education,chat,analytics_dashboard,account_access,medium,in_progress,standard,I cannot log in; the system says my password i...,Sorry to hear you're having trouble accessing ...,NaN,NaN,0,positive,5,1,android,LATAM
4,TCKT_000005,2023-08-27T16:08:33,CUST_09556,enterprise,phone_transcript,login_auth,billing_problem,low,resolved,gold,My invoice amount is incorrect compared to the...,Thanks for reaching out about the billing issu...,Adjusted the invoice and issued a refund where...,61.32,0,very_negative,2,0,web,NaN


In [3]:
df.tail()

,ticket_id,created_at,customer_id,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,has_attachment,platform,region
99995,TCKT_099996,2023-05-16T06:23:50,CUST_06828,education,in_app,analytics_dashboard,billing_problem,high,resolved,standard,My invoice amount is incorrect compared to the...,Thanks for reaching out about the billing issu...,Adjusted the invoice and issued a refund where...,7.97,0,neutral,0,0,desktop_app,EU
99996,TCKT_099997,2025-03-10T04:46:24,CUST_06621,education,in_app,billing,bug,low,resolved,standard,I am seeing an error message when I click on b...,Thanks for reporting this bug. We will look in...,Patched the bug and deployed a fix to production.,16.48,0,negative,2,0,api_client,MEA
99997,TCKT_099998,2025-11-07T19:33:01,CUST_03719,non_profit,chat,billing,bug,low,resolved,standard,I am seeing an error message when I click on b...,Thanks for reporting this bug. We will look in...,Patched the bug and deployed a fix to production.,56.99,0,very_positive,5,0,web,LATAM
99998,TCKT_099999,2022-06-20T22:29:10,CUST_02290,small_business,email,analytics_dashboard,account_access,low,open,standard,I cannot log in; the system says my password i...,Sorry to hear you're having trouble accessing ...,NaN,NaN,0,neutral,0,1,api_client,NaN
99999,TCKT_100000,2022-02-27T14:33:20,CUST_06371,education,email,data_export,security_concern,low,closed_no_action,standard,We need details about your data encryption and...,We take security very seriously. We will look ...,Ticket closed without further action after no ...,230.21,0,neutral,0,0,desktop_app,APAC


In [4]:
df["sla_plan"].value_counts()

sla_plan
standard    60066
gold        29875
platinum    10059
Name: count, dtype: int64

In [6]:
df["region"].value_counts()

region
MEA      20151
EU       19960
LATAM    19959
APAC     19933
Name: count, dtype: int64

In [7]:
df["status"].value_counts()

status
resolved            50131
in_progress         19783
on_hold             10165
closed_no_action     9982
open                 9939
Name: count, dtype: int64

In [8]:
df["resolution_time_hours"].max()


239.96

In [9]:
df["resolution_time_hours"].min()

0.11

In [10]:
df["resolution_time_hours"].mode()

0     6.95
1     8.36
2    13.65
3    17.48
4    22.38
Name: resolution_time_hours, dtype: float64

In [19]:
df["has_attachment"].value_counts()

has_attachment
0    75048
1    24952
Name: count, dtype: int64

In [21]:
df["resolution_time_hours"].mean()

45.01312295177415

In [23]:
df["reopened"].value_counts()

reopened
0    94955
1     5045
Name: count, dtype: int64

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ticket_id              100000 non-null  object 
 1   created_at             100000 non-null  object 
 2   customer_id            100000 non-null  object 
 3   customer_segment       100000 non-null  object 
 4   channel                100000 non-null  object 
 5   product_area           100000 non-null  object 
 6   issue_type             100000 non-null  object 
 7   priority               100000 non-null  object 
 8   status                 100000 non-null  object 
 9   sla_plan               100000 non-null  object 
 10  initial_message        100000 non-null  object 
 11  agent_first_reply      100000 non-null  object 
 12  resolution_summary     60113 non-null   object 
 13  resolution_time_hours  60113 non-null   float64
 14  reopened               100000 non-nul

In [27]:
# Filling in the missing values and removing unnecessary features [data cleaning & preprocessing]

df = df.drop(columns=[
    "region",
    "resolution_summary",
    # identifiers and text, not useful for predicting
    "ticket_id",
    "customer_id",
    "created_at",
    "initial_message",
    "agent_first_reply",
    # only known AFTER the ticket is handled, so the model can't use them in real life
    "customer_sentiment",
    "csat_score",
])

In [29]:
# Checking the missing values
# resolution_time_hours is only missing for tickets that aren't finished yet (open, in_progress, on_hold)
# so I'm not filling it with random numbers, I just only use finished tickets later on

df.isnull().sum()

customer_segment             0
channel                      0
product_area                 0
issue_type                   0
priority                     0
status                       0
sla_plan                     0
resolution_time_hours    39887
reopened                     0
has_attachment               0
platform                     0
dtype: int64

In [31]:
df["issue_type"].value_counts()

issue_type
how_to              12739
account_access      12609
performance         12513
feature_request     12487
other               12463
security_concern    12458
billing_problem     12382
bug                 12349
Name: count, dtype: int64

In [33]:
df = pd.get_dummies(
    df,
    columns=["issue_type", "priority", "status"],
    dtype=int
)

sla_plan = {
    "platinum": 1,
    "gold": 2,
    "standard": 3,
}
df["sla_plan"] = df["sla_plan"].map(sla_plan)

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 25 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   customer_segment             100000 non-null  object 
 1   channel                      100000 non-null  object 
 2   product_area                 100000 non-null  object 
 3   sla_plan                     100000 non-null  int64  
 4   resolution_time_hours        60113 non-null   float64
 5   reopened                     100000 non-null  int64  
 6   has_attachment               100000 non-null  int64  
 7   platform                     100000 non-null  object 
 8   issue_type_account_access    100000 non-null  int32  
 9   issue_type_billing_problem   100000 non-null  int32  
 10  issue_type_bug               100000 non-null  int32  
 11  issue_type_feature_request   100000 non-null  int32  
 12  issue_type_how_to            100000 non-null  int32  
 13  

In [37]:
df.head()

,customer_segment,channel,product_area,sla_plan,resolution_time_hours,reopened,has_attachment,platform,issue_type_account_access,issue_type_billing_problem,...,issue_type_security_concern,priority_high,priority_low,priority_medium,priority_urgent,status_closed_no_action,status_in_progress,status_on_hold,status_open,status_resolved
0,individual,email,data_export,3,36.53,0,0,android,1,0,...,0,0,1,0,0,0,0,0,0,1
1,individual,in_app,billing,3,238.32,0,0,web,0,0,...,1,0,0,1,0,1,0,0,0,0
2,small_business,chat,api_integration,3,NaN,0,0,android,0,0,...,0,0,1,0,0,0,1,0,0,0
3,education,chat,analytics_dashboard,3,NaN,0,1,android,1,0,...,0,0,0,1,0,0,1,0,0,0
4,enterprise,phone_transcript,login_auth,2,61.32,0,0,web,0,1,...,0,0,1,0,0,0,0,0,0,1


In [39]:
df["status_resolved"].value_counts()

status_resolved
1    50131
0    49869
Name: count, dtype: int64

In [41]:
# Defining the new column before setting it as a target

# Only resolved tickets can be judged as breached or not
resolved = df[df["status_resolved"] == 1].copy()

# sla_plan was mapped earlier: platinum = 1, gold = 2, standard = 3
# these hours are example values since we are dealing with synthetic data
sla_hours = {1: 8, 2: 16, 3: 24}

# breach = the ticket took longer than its plan allows
resolved["sla_breach"] = (
    resolved["resolution_time_hours"] > resolved["sla_plan"].map(sla_hours)
).astype(int)

resolved["sla_breach"].value_counts(normalize=True)

sla_breach
1    0.581098
0    0.418902
Name: proportion, dtype: float64

In [43]:
# Defining the features as well as the target
# Left out on purpose: resolution_time_hours (used to make the target), reopened and status (only known after the ticket is handled)

X = resolved[[
     "customer_segment",
     "channel",
     "product_area",
     "platform",
     "issue_type_account_access",
     "issue_type_billing_problem", 
     "issue_type_bug", 
     "issue_type_feature_request", 
     "issue_type_how_to",  
     "issue_type_other",
     "issue_type_performance", 
     "issue_type_security_concern",   
     "priority_high",
     "priority_low",  
     "priority_medium", 
     "priority_urgent",  
     "sla_plan",
     "has_attachment"
]]

In [45]:
# Defining the target (Y)
# (X stays as defined above, the old line X = df.drop(columns=["sla_breach"]) put the answer columns back in)

y = resolved["sla_breach"]

In [47]:
# Conducting a train_test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [48]:
# Checking the train/test split

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (35091, 18)
Testing features: (15040, 18)
Training target: (35091,)
Testing target: (15040,)


In [49]:
# Training the model

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Identify column types
categorical_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

# Preprocess numerical columns
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocess categorical columns
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# Create the complete model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Train the model
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [50]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("Precision:", precision_score(
    y_test, y_pred, zero_division=0
))

print("Recall:", recall_score(
    y_test, y_pred, zero_division=0
))

print("F1-score:", f1_score(
    y_test, y_pred, zero_division=0
))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7505984042553191
Precision: 0.8086870436827125
Recall: 0.7477116704805492
F1-score: 0.7770049343083051

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.75      0.72      6300
           1       0.81      0.75      0.78      8740

    accuracy                           0.75     15040
   macro avg       0.75      0.75      0.75     15040
weighted avg       0.76      0.75      0.75     15040


Confusion Matrix:
[[4754 1546]
 [2205 6535]]


In [51]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[4754 1546]
 [2205 6535]]


In [52]:
# Comparing against a "dumb" model that always guesses the most common answer
# my model has to beat this, otherwise it hasn't learned anything
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

print("Dummy accuracy:", accuracy_score(y_test, dummy_pred))
print("Dummy recall:", recall_score(y_test, dummy_pred, zero_division=0))
print("Model accuracy:", accuracy_score(y_test, y_pred))
print("Model recall:", recall_score(y_test, y_pred, zero_division=0))

Dummy accuracy: 0.5811170212765957
Dummy recall: 1.0
Model accuracy: 0.7505984042553191
Model recall: 0.7477116704805492


In [53]:
# Testing the model on 5 different splits of the data to see if the score is stable
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5, scoring="roc_auc")
print("ROC-AUC for each split:", scores.round(3))
print("Average:", scores.mean().round(3))

ROC-AUC for each split: [0.824 0.827 0.827 0.832 0.83 ]
Average: 0.828


In [54]:
# Checking which features the model relies on most (the leakage check)
# if one feature controls everything, something is still leaking
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]

importance = pd.Series(coefficients, index=feature_names).sort_values(key=abs, ascending=False)
importance.head(10)

num__priority_urgent                    -1.852993
num__priority_low                        0.931359
num__sla_plan                           -0.871124
num__priority_high                      -0.622561
num__priority_medium                     0.411268
cat__channel_in_app                     -0.072237
cat__product_area_analytics_dashboard   -0.072164
cat__platform_desktop_app               -0.068552
cat__product_area_notifications         -0.056060
cat__customer_segment_education         -0.055949
dtype: float64

In [55]:
# Testing the model on a brand new ticket I make up myself
# the columns must match that of X exactly
new_ticket = pd.DataFrame([{
    "customer_segment": "enterprise",
    "channel": "email",
    "product_area": "billing",
    "platform": "web",
    "issue_type_account_access": 0,
    "issue_type_billing_problem": 1,
    "issue_type_bug": 0,
    "issue_type_feature_request": 0,
    "issue_type_how_to": 0,
    "issue_type_other": 0,
    "issue_type_performance": 0,
    "issue_type_security_concern": 0,
    "priority_high": 0,
    "priority_low": 1,
    "priority_medium": 0,
    "priority_urgent": 0,
    "sla_plan": 1,          # 1 = platinum, 2 = gold, 3 = standard
    "has_attachment": 0
}])

print("Predicted class (1 = breach):", model.predict(new_ticket)[0])
print("Chance of breach:", model.predict_proba(new_ticket)[0][1].round(3))

Predicted class (1 = breach): 1
Chance of breach: 0.964


In [58]:
# Testing the model on real tickets from the test set, side by side with the truth
y_proba = model.predict_proba(X_test)[:, 1]

results = X_test.copy()
results["actual_sla_breach"] = y_test
results["predicted_sla_breach"] = y_pred
results["chance_of_sla_breach"] = y_proba.round(3)

results[["sla_plan", "priority_low", "priority_urgent", "actual_sla_breach", "predicted_sla_breach", "chance_of_sla_breach"]].head(10)

,sla_plan,priority_low,priority_urgent,actual_sla_breach,predicted_sla_breach,chance_of_sla_breach
21375,2,1,0,1,1,0.874
25407,3,0,0,0,0,0.056
18992,3,0,0,0,0,0.434
45630,2,0,0,0,0,0.200
23165,3,0,0,0,0,0.063
63258,3,0,0,0,0,0.064
57891,3,1,0,1,1,0.711
80690,3,0,0,1,0,0.436
2743,3,0,0,1,0,0.419
17882,2,0,0,0,0,0.205
